[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/08_langgraph_production_track.ipynb)


# Agentic Systems Foundations
## Notebook 08: The Production Track — LangGraph, LangChain, LangSmith
**Duration:** appendix &nbsp;|&nbsp; **Mode:** Self-paced / take-home

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** all of it — rebuilt on the stack you would actually deploy.


In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Makes `agent_core` importable whether you are in Colab, in a local venv,
# or running from a clone. Installs nothing you do not need: the package's
# only hard dependency is the Python standard library.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork


def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


if IN_COLAB:
    # openai for the real provider; jsonschema + langchain for the parallel
    # mappings shown in notebook 03. All optional — the notebook degrades
    # gracefully if any is missing.
    _pip("openai", "python-dotenv", "jsonschema", "langchain-core")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

# Where the Acme data lives — the tools resolve this automatically, but we
# print it so a path problem is visible immediately rather than as an empty
# search result three cells later.
from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDER  (works with NO key at all)
# ============================================================
# Default stack = OpenAI gpt-4o-mini with native tool calling.
# In Colab the key is read from the SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON -> re-run this cell.
#
# With NO key we fall back to MockToolCallLLM. Read that name literally:
# unlike a text-only mock, it DECIDES TOOL CALLS, so the entire agent loop —
# every notebook in this session — runs offline and deterministically.
import os

def _load_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

HAS_KEY = _load_key()
os.environ.setdefault("AGENT_LLM_PROVIDER", "openai" if HAS_KEY else "mock")

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
if not HAS_KEY:
    print("\nNo API key found -> running on the offline mock.")
    print("Everything in this notebook still works. Outputs are labelled [mock].")

> ### This notebook is an APPENDIX
> It sits **outside** the 180-minute session clock. Notebooks 01–07 teach the
> mechanics from scratch; this one rebuilds the same agent on **LangGraph +
> LangChain + LangSmith** so you leave with something you can put in a repo.
>
> **It requires `OPENAI_API_KEY`.** That is deliberate — the point of this track
> is to show real production behaviour, not a simulation of it. Notebooks 01–07
> remain fully runnable with no key.

## WHY a second implementation at all

Because both halves of the following are true, and holding only one of them is
how teams get hurt:

1. **You should not hand-roll an agent framework in production.** Checkpointing,
   streaming, retries, observability, human-in-the-loop interrupts — that is a
   large amount of well-solved infrastructure, and reimplementing it is a bad use
   of your team.

2. **A framework will not make any of your design decisions for you.** It will
   not choose your enum values, write a description that says *when* to use a
   tool, decide what your tool returns when it finds nothing, or notice that your
   agent called the same tool five times.

Notebooks 01–07 taught you #2. This notebook gives you #1 — and shows you that
every lesson from #2 survives the move intact.


## The translation table

| `agent_core` (learn the mechanism) | `agent_lc` (ship it) |
|---|---|
| `while not done:` | `StateGraph` edges |
| `AgentState` dataclass | `TypedDict` + **`add_messages` reducer** |
| `llm.decide()` | the `agent` node, `model.bind_tools()` |
| `ToolRegistry.dispatch()` | `ToolNode(handle_tool_errors=True)` |
| `build_schema()` from docstrings | Pydantic `args_schema` |
| `validate_args()` | Pydantic validation |
| `TerminationPolicy` | conditional edge + `recursion_limit` |
| `Skill` + `Router` | scoped subgraphs + a supervisor |
| `Trace` | LangSmith run trees |
| *(we had nothing)* | **checkpointers** — memory, resumption, interrupts |

Two rows deserve a second look — one where the framework is genuinely better,
one where it genuinely is not. We get to both below.


In [ ]:
# ============================================================
# LANGGRAPH TRACK — needs OPENAI_API_KEY
# ============================================================
# Everything above ran offline on the mock. From here we use a REAL model,
# because this half of the session is about what you actually deploy.
# Without a key these cells skip cleanly — the from-scratch cells above have
# already made the conceptual point.
import os

LC_READY = bool(os.getenv("OPENAI_API_KEY"))
if LC_READY:
    from langchain_openai import ChatOpenAI
    chat = ChatOpenAI(model=os.getenv("AGENT_LLM_MODEL", "gpt-4o-mini"), temperature=0)
    print("LangGraph track: ready ->", chat.model_name)
else:
    chat = None
    print("LangGraph track: SKIPPED (no OPENAI_API_KEY).")
    print("Set a key to run these cells. The from-scratch cells above still ran.")

## 1. Tools — Pydantic instead of docstring reflection

`agent_core` derived schemas from signatures and docstrings using about a hundred
lines of hand-rolled reflection. Here it is a Pydantic model. **Same contract,
declared instead of inferred.**


In [ ]:
# No key needed — schemas are declarative.
import json
from agent_lc.tools_lc import ACME_TOOLS, RefundArgs

print("The Pydantic model IS the schema:")
print(json.dumps(RefundArgs.model_json_schema()["properties"], indent=1))

Everything the from-scratch package encoded by convention is now a typed,
validated field:

- `Literal[...]` → `enum` — **identical in both**, and still the highest-value
  line you can write
- our `pattern:` docstring marker → `Field(pattern=r"^ACME-\d{4}$")` — a real
  constraint object rather than something we parse out of prose
- our `validate_args()` → Pydantic's validator: faster, standards-complete,
  tested by a very large number of people

**What Pydantic did not do:** choose the four reason codes, decide that an order
ID looks like `ACME-####`, or write a description that says *when* to use the
tool. Those are still yours, and they are the part that determines whether the
model calls your tool correctly.


## 2. The loop — as a graph

Here is the row where **LangGraph is genuinely better than what we built**.

The agenda asked for *"explicit state representation and updates"*. In the
from-scratch loop, state was explicit because we were disciplined — and notebook
02 showed what happens when you forget the write-back.

In LangGraph you **declare** it:

```python
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

`add_messages` is a **reducer**: it says *how this field updates*. A node returns
`{"messages": [reply]}` and the reducer appends. The single most important line
in notebook 02 is now a property of the schema, applied to every node
automatically.

You can still get it wrong — return the wrong key, choose the wrong reducer — but
you **cannot silently omit it**.


In [ ]:
if not LC_READY:
    print('skipped — needs OPENAI_API_KEY')
else:
    \
    from langchain_core.messages import HumanMessage
    from agent_lc import build_agent_graph, draw, show_messages, call_sequence

    graph = build_agent_graph(
        chat,
        system_prompt="You answer using tools. Gather evidence before answering.",
        max_steps=6,
    )
    print(draw(graph))          # the cycle: agent -> tools -> agent

In [ ]:
if not LC_READY:
    print('skipped — needs OPENAI_API_KEY')
else:
    \
    # The invoke payload IS the initial state.
    goal = "Order ACME-1046 — I changed my mind and want a refund. Am I eligible?"
    out = graph.invoke({"messages": [HumanMessage(goal)], "steps": 0, "stop_reason": None})

    print("steps :", out["steps"])
    print("tools :", " -> ".join(call_sequence(out)))
    print()
    print(show_messages(out))

That transcript is the **same shape** as `state.messages` from notebook 02 —
human, ai-with-tool-calls, tool, ai, … — built by a reducer instead of by
`add_message()` calls. That is the only difference.


> ### ✋ Predict before you run
> Next we run the identical goal through **both engines** and compare the traces. **Will the trajectories match?** If they differ, is that a bug in one of them?
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
if not LC_READY:
    print('skipped — needs OPENAI_API_KEY')
else:
    \
    # Both engines, one comparison table. `to_trace()` converts a LangGraph result
    # into the Trace type from notebook 05 — possible precisely because both record
    # the same underlying events.
    import os
    from agent_core import Agent, compare
    from agent_lc import to_trace

    os.environ["AGENT_LLM_PROVIDER"] = "openai"      # fair comparison: same model both sides

    scratch_trace = Agent().run(goal).trace
    lc_trace = to_trace(
        graph.invoke({"messages": [HumanMessage(goal)], "steps": 0, "stop_reason": None}),
        goal,
    )
    print(compare({"from scratch": scratch_trace, "langgraph": lc_trace}))

**On any trajectory differences:** neither is a bug. The two tracks build
slightly different system prompts and hand the model a slightly different tool
inventory, and the model is free to sequence its calls differently. That is the
defining property from notebook 01 — *the path is decided at runtime* — showing
up as a fact about your own two implementations.

It is also why the last line of notebook 01 matters: **you cannot test an agent
by asserting on one output.** Assert on trajectories, over a suite.


## 3. Termination — where the framework does NOT save you

This is the row where you must not assume the framework has it covered.

LangGraph gives you **`recursion_limit`**: a hard backstop that stops a runaway
graph. What it tells you is *"the ceiling was hit"*. What it never tells you is
*"the agent called the same tool with the same arguments five times"*.

So adopting LangGraph does **not** give you the thing notebook 05 argues is the
actual engineering. It gives you the safety net and leaves the diagnosis to you
— and because `recursion_limit` *looks* like it covers the problem, the
diagnostic conditions are the thing teams most often never write.

The good news: everything in `agent_core/control.py` ports directly, because it
was never framework-specific. It is reasoning over the message history.


In [ ]:
# The port, run on a test double so the comparison is reproducible and free.
# (agent_lc.fake_model is for tests, not teaching — see its docstring.)
from langchain_core.messages import HumanMessage
from agent_lc.graph import build_agent_graph, build_diagnostic_graph
from agent_lc.fake_model import FakeToolCallingModel

g = "What is the status of order ACME-1042?"
start = lambda: {"messages": [HumanMessage(g)], "steps": 0, "stop_reason": None}

budget_only = build_agent_graph(FakeToolCallingModel(fault="loop_forever"), max_steps=8)
diagnostic  = build_diagnostic_graph(FakeToolCallingModel(fault="loop_forever"), max_steps=8)

b = budget_only.invoke(start())
d = diagnostic.invoke(start())
print(f"budget-only  steps={b['steps']}  stop={b['stop_reason']}")
print(f"diagnostic   steps={d['steps']}  stop={d['stop_reason']}")
print()
print("Same lesson as notebook 05, same numbers, different engine.")

## 4. Skills — scoped subgraphs and a supervisor

The scoping argument from notebook 04 was never about our implementation. Tool
choice degrades as the list grows because of how *models* behave, so it survives
the change of engine completely.

What production does differently is *how it routes*: not keyword scoring, but a
cheap model with **structured output**.


In [ ]:
# Keyword routing — identical to notebook 04, no key needed.
from agent_lc import KeywordSupervisor, acme_lc_skills

skills = acme_lc_skills()
for s in skills:
    print(f"{s.name:<20} {len(s.tools)} tools: {', '.join(t.name for t in s.tools)}")
print()
print(KeywordSupervisor(skills).explain("I want a refund on ACME-1046, I changed my mind."))

In [ ]:
if not LC_READY:
    print('skipped — needs OPENAI_API_KEY')
else:
    \
    # LLM routing, constrained by structured output.
    #
    # WHY structured output and not 'reply with the skill name': parsing a name out
    # of prose is a classic source of routing flakiness. Constrain the model to a
    # Literal of real skill names and that whole failure mode disappears.
    from agent_lc import build_supervisor_graph, call_sequence, final_answer

    supervisor = build_supervisor_graph(chat)

    for g2 in ["I want a refund on ACME-1046, I changed my mind.",
               "How much does the Growth plan cost per month?",
               "What is the status of order ACME-1048?"]:
        r = supervisor.invoke({"messages": [HumanMessage(g2)],
                               "steps": 0, "stop_reason": None, "skill": None})
        print(f"{r['skill']:<20} {' -> '.join(call_sequence(r)) or '(none)'}")
        print(f"   {r['stop_reason']}")

**The invariant:** whichever router you use, the executing agent still sees
**only its own skill's tools**. That scoping is what buys the reliability.
Swapping a keyword scorer for an LLM classifier improves *which* skill gets
picked — not what happens after.

**What LangGraph adds that we did not have:** real **handoffs**. Because skills
are graph nodes, a skill can route onward to another with state carried across.
That is the honest answer to *"what if a request spans two jobs?"* — at scale you
neither compose everything nor accept a half-answer; you hand off.


## 5. Memory — capability we genuinely did not have

A **checkpointer** persists graph state per `thread_id`. Swap `InMemorySaver` for
Postgres or Redis and your agent survives a process restart mid-conversation.

This is a fair reason to adopt the framework. Serialisation, concurrency,
resumption after a crash — that is real work you should not redo, and it is
nowhere in `agent_core`.


In [ ]:
if not LC_READY:
    print('skipped — needs OPENAI_API_KEY')
else:
    \
    from agent_lc import build_conversational

    conv = build_conversational(chat, system_prompt="You answer using tools.")
    thread = {"configurable": {"thread_id": "customer-8891"}}

    first = conv.invoke({"messages": [HumanMessage("What is the status of order ACME-1048?")]}, thread)
    print("Q1:", final_answer(first)[:200])
    print()

    # No order ID in the second question. It works because the checkpointer kept the
    # thread — this is memory ACROSS invocations, not just within one run.
    second = conv.invoke({"messages": [HumanMessage("And what plan is that on?")]}, thread)
    print("Q2:", final_answer(second)[:200])
    print()
    print("messages retained on the thread:", len(second["messages"]))

## 6. Observability — LangSmith

`agent_core/trace.py` argued that tracing is the debugging interface for the
whole paradigm, then implemented a printed string. Right for teaching — you can
read all of it. Wrong for production, where you need search, diffing across runs,
cost breakdowns, and a link you can paste into a ticket.

Turning LangSmith on is **two environment variables**. No code changes, no
decorators — LangChain instruments its own primitives:

```python
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "..."
```

What it gives you over a printed trace:

- **persistence and search** — *"every run last week where the agent escalated"*
  becomes a query, not a grep through notebook output
- **cost and latency per step**, aggregated across runs
- **diffing** two runs of the same task after a prompt change
- **datasets and evaluators** — `data/tasks/agent_tasks.jsonl` becomes a
  LangSmith dataset you run on every commit
- **a shareable URL** — notebook 05's "trace as a bug report", except your
  colleague clicks a link

What it does *not* give you: the judgement about **which conditions to check and
in what order**. That is still yours.


In [ ]:
from agent_lc import enable_langsmith, langsmith_status
enable_langsmith(project="agentic-systems-foundations")
print(langsmith_status())

## 7. `create_react_agent` — what you would actually write

Everything above, in two lines. Having read `agent_lc/graph.py`, you know exactly
what it builds — which is the only reason it is safe to use.


In [ ]:
if not LC_READY:
    print('skipped — needs OPENAI_API_KEY')
else:
    \
    from agent_lc import build_prebuilt_agent

    agent = build_prebuilt_agent(chat, system_prompt="You answer using tools.")
    out = agent.invoke({"messages": [HumanMessage(
        "Order ACME-1046 — I changed my mind and want a refund. Am I eligible?")]})

    print("tools :", " -> ".join(call_sequence(out)))
    print("answer:", final_answer(out)[:320])

### What you still own after adopting the prebuilt agent

- the tool **descriptions and schemas** — nothing writes those for you
- the **system prompt**
- **which tools are in scope** for each job
- `recursion_limit`, and **every diagnostic termination condition beyond it**
- whether tool errors are handled or fatal (`handle_tool_errors`)
- what your tools return when they find nothing

That list is the entire content of notebooks 01–07. **None of it was made
obsolete by adopting a framework** — which is exactly why we built it from
scratch first.


## Recap

- **Ship the framework.** Checkpointing, streaming, retries, observability and
  human-in-the-loop are solved infrastructure; reimplementing them is a poor use
  of your team.
- **LangGraph is genuinely better in one place we care about:** state is a
  *declared schema with reducers*, so the write-back cannot be silently omitted.
- **It is genuinely not better in another:** `recursion_limit` is a backstop, not
  a diagnosis. Repetition, error streaks and no-new-information are still yours
  to write — and are the ones most often skipped, because the backstop looks like
  it covers them.
- **Every design decision transferred unchanged:** enums, patterns, descriptions
  that say *when*, scoping, prose returns, grounding, negative tests.

> The framework abstracts the mechanics, not the design decisions.

**Where to go next:** `scripts/check_langgraph.py` verifies this whole track and
doubles as a worked example of testing a graph. The capstone in
`teaching/exercises.md` can be built on either track — try it on this one.
